# CercaFungo — Esplorazione Dataset

Notebook per esplorare i dati scaricati, verificare le annotazioni,
e analizzare la qualita' del dataset prima del training.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import yaml
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

# Carica config
ROOT = Path('..').resolve()
with open(ROOT / 'config.yaml') as f:
    config = yaml.safe_load(f)

print(f'Root: {ROOT}')
print(f'Specie configurate: {len(config["species"])}')

## 1. Panoramica Specie

In [ ]:
import pandas as pd

species_df = pd.DataFrame(config['species'])
species_df[['id', 'italian', 'latin', 'edible', 'toxic', 'season', 'habitat']]

## 2. Verifica Dati Scaricati

In [ ]:
# Controlla quante immagini abbiamo per sorgente
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

sources = {
    'Roboflow': ROOT / config['paths']['raw_images'] / 'roboflow',
    'iNaturalist': ROOT / config['paths']['raw_images'] / 'inaturalist',
    'Sintetiche': ROOT / config['paths']['synthetic_images'],
    'Manuali': ROOT / config['paths']['raw_images'] / 'manual',
}

for name, path in sources.items():
    if path.exists():
        count = sum(1 for f in path.rglob('*') if f.suffix.lower() in IMAGE_EXT)
        print(f'{name}: {count} immagini ({path})')
    else:
        print(f'{name}: directory non trovata ({path})')

## 3. Visualizza Campioni per Specie (iNaturalist)

In [ ]:
inat_dir = ROOT / config['paths']['raw_images'] / 'inaturalist'

if inat_dir.exists():
    species_dirs = [d for d in inat_dir.iterdir() if d.is_dir()]
    
    for species_dir in species_dirs:
        images = [f for f in species_dir.iterdir() if f.suffix.lower() in IMAGE_EXT]
        if not images:
            continue
        
        # Mostra 4 campioni per specie
        fig, axes = plt.subplots(1, min(4, len(images)), figsize=(16, 4))
        if not hasattr(axes, '__len__'):
            axes = [axes]
        
        for ax, img_path in zip(axes, images[:4]):
            img = cv2.imread(str(img_path))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis('off')
        
        plt.suptitle(f'{species_dir.name} ({len(images)} immagini)', fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print('Directory iNaturalist non trovata. Esegui download_inaturalist.py')

## 4. Statistiche Dataset Processato

In [ ]:
from dataset.visualize import compute_dataset_statistics, print_statistics

processed_dir = ROOT / config['paths']['processed_dataset']
if processed_dir.exists():
    print_statistics(processed_dir)
else:
    print(f'Dataset processato non trovato: {processed_dir}')
    print('Esegui: python -m dataset.prepare_dataset')

## 5. Preview Annotazioni

In [ ]:
from dataset.visualize import preview_annotations

if processed_dir.exists():
    preview_annotations(processed_dir, split='train', num_images=8)
    plt.show()

## 6. Distribuzione BBox

In [ ]:
from dataset.visualize import plot_bbox_distribution

if processed_dir.exists():
    plot_bbox_distribution(processed_dir)
    plt.show()

## 7. Analisi Augmentations

In [ ]:
from utils.augmentations import (
    get_forest_detection_augmentation,
    apply_synthetic_shadow,
    apply_dappled_light,
)

# Prendi un'immagine di esempio
sample_dir = processed_dir / 'train' / 'images' if processed_dir.exists() else None
sample_img = None

if sample_dir and sample_dir.exists():
    for f in sample_dir.iterdir():
        if f.suffix.lower() in IMAGE_EXT:
            sample_img = cv2.imread(str(f))
            break

if sample_img is not None:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes[0][0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
    axes[0][0].set_title('Originale')
    
    augment = get_forest_detection_augmentation(640)
    
    for i in range(1, 8):
        r, c = divmod(i, 4)
        try:
            result = augment(
                image=cv2.resize(sample_img, (640, 640)),
                bboxes=[],
                class_labels=[]
            )
            axes[r][c].imshow(cv2.cvtColor(result['image'], cv2.COLOR_BGR2RGB))
        except:
            pass
        axes[r][c].set_title(f'Augmentation {i}')
        axes[r][c].axis('off')
    
    axes[0][0].axis('off')
    plt.suptitle('Augmentation Pipeline', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Nessuna immagine di esempio trovata.')